# ZelusBench: Attention Updating

**Can the model update its representation after geometric transforms?**

This task applies rotations, translations, and other transforms mid-scenario
and checks whether the model correctly updates all affected points.
The model must propagate changes through the dependency DAG.

Transform levels:
- **Static** (0): no transforms — baseline
- **Light** (1-2): a rotation or translation
- **Heavy** (3-5): multiple transforms stacked
- **Extreme** (6+): heavy transform load

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import re
import os

os.environ["RENDER_SUBRUNS"] = "False"

In [ ]:
from zelusbench.scenarios.config import (
    ScenarioConfig, DAGStructure, QueryType,
    DistractorLevel, TransformDensity,
)
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.parser import parse_model_response
from zelusbench.evaluation.scorer import score_query


def score_response(response_text, scenario):
    query_dicts = [q.to_dict() for q in scenario.queries]
    parts = re.split(r'\[Query\s+q_\d+\]', response_text)
    if len(parts) > 1:
        parts = parts[1:]
    if len(parts) != len(query_dicts):
        parts = [response_text] * len(query_dicts)
    return [score_query(qd, parse_model_response(rp, qd)) for qd, rp in zip(query_dicts, parts)]

In [ ]:
TRANSFORM_LEVELS = {
    0: TransformDensity.STATIC,
    2: TransformDensity.LIGHT,
    4: TransformDensity.HEAVY,
    7: TransformDensity.EXTREME,
}

@kbench.task(store_task=False)
def updating_scenario(llm, num_transforms: int, seed: int) -> float:
    """Run one scenario with a given number of mid-sequence transforms."""
    cfg = ScenarioConfig(
        dim=3,
        min_chain_depth=4, max_chain_depth=6,
        dag_structure=DAGStructure.LINEAR,
        distractor_level=DistractorLevel.CLEAN,
        transform_density=TRANSFORM_LEVELS[num_transforms],
        transform_types=["rotation", "translation"],
        query_types=[QueryType.POSITION, QueryType.DISTANCE],
        num_queries=3, num_splits=3,
        seed=seed,
    )
    gen = ScenarioGenerator(cfg)
    scenario = gen.generate(f"updating_t{num_transforms}_s{seed}")

    response = llm.prompt(scenario.prompt)
    scores = score_response(response, scenario)
    avg = float(np.mean([s.score for s in scores]))

    for s in scores:
        kbench.assertions.assert_true(
            s.score > 0,
            expectation=(
                f"{s.query_id} (transforms={num_transforms}): "
                f"model should update after transforms. Tier={s.tier.name}"
            )
        )
    return avg

In [ ]:
eval_data = pd.DataFrame([
    {"num_transforms": t, "seed": seed}
    for t in [0, 2, 4, 7]
    for seed in range(5)
])

print(f"Evaluation matrix: {len(eval_data)} scenarios")
print(f"Transform counts: {sorted(eval_data.num_transforms.unique())}")

In [ ]:
@kbench.task(name="zelusbench_attention_updating")
def zelusbench_attention_updating(llm) -> tuple[float, float]:
    """Attention updating: accuracy before vs after transforms.

    Returns (mean_accuracy, std_dev).
    """
    with kbench.client.enable_cache():
        runs = updating_scenario.evaluate(
            llm=[llm],
            evaluation_data=eval_data,
            n_jobs=2,
            timeout=180,
            remove_run_files=True,
            stop_condition=lambda r: len(r) == len(eval_data),
            max_attempts=60,
            retry_delay=10,
        )

    results_df = runs.as_dataframe()
    scores = results_df["result"].dropna().tolist()
    accuracy = float(np.mean(scores)) if scores else 0.0
    std = float(np.std(scores)) if scores else 0.0

    kbench.assertions.assert_true(
        len(scores) > 0,
        expectation="At least some scenarios should produce results"
    )
    return accuracy, std

In [ ]:
run = zelusbench_attention_updating.run(kbench.llm)
run

In [ ]:
%choose zelusbench_attention_updating